# ImageNet100 用の DeepInversion+ ノートブック

In [1]:
import os
import sys
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



In [2]:
# 使用するgpuを指定
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

## パスの設定

In [3]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"


# tiny-imagenetのbaseline用パス
base_cifar100_path = "baseline/imagenet100"

# baseline
method = "baseline_mu"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_1.0")

## 色々と設定

In [4]:

# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)


from utils import factory
import models


# --- 1) 設定を記述した jsonファイル の内容を読む ---
with open(os.path.join(project_root, "exps", "BASELINE", "imnet100.json")) as f:
    args = json.load(f)

args["device"] = ["0"]            # 必要に応じて
# args["model_name"] = "baseline" 


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")       # 読み込むモデルの指定

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# fc層の出力次元数を変更
net.update_fc(num_outputs)

# state_dict の読み込み
net.load_state_dict(state_dict)

# protos, forget_classes も保存されていれば復元
if "protos" in ckpt:
    net._protos = ckpt["protos"]
    # assert False
if "forget_classes" in ckpt and hasattr(net, "forget_classes"):
    net.forget_classes = ckpt["forget_classes"]

net.cuda().eval()

# 忘却クラスや class_order を取り出す
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get("_class_order", None)
print(class_order)

<ipython-input-4-e66ad6e43c64>:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

## Hookの設定

In [5]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


## 最適化対象の準備など

In [32]:
# exp_name = args.exp_name
# # final images will be stored here:
# adi_data_path = "./final_images/%s"%exp_name
# # temporal data and generations will be stored here
# exp_name = "generations/%s"%exp_name

iterations = 2000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 120
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"

# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# ver1からr_featureを増加
# coefficients["r_feature"] = 0.03   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# ver3からr_featureを増加
# coefficients["r_feature"] = 0.1   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"


# ver1 から main_loss_multiplie を少し増加
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# ver1 から main_loss_multiplie を少し増加
coefficients["r_feature"] = 0.01   # github由来
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.0005
coefficients["l2"] = 0.00001
coefficients["lr"] = 0.25
coefficients["main_loss_multiplier"] = 1.2
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v6"

network_output_function = lambda x: x


prefix = "runs/data_generation_di+_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [8]:
def lr_policy(lr_fn):
    def _alr(optimizer, iteration, epoch):
        lr = lr_fn(iteration, epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

    return _alr


def lr_cosine_policy(base_lr, warmup_length, epochs):
    def _lr_fn(iteration, epoch):
        if epoch < warmup_length:
            lr = base_lr * (epoch + 1) / warmup_length
        else:
            e = epoch - warmup_length
            es = epochs - warmup_length
            lr = 0.5 * (1 + np.cos(np.pi * e / es)) * base_lr
        return lr

    return lr_policy(_lr_fn)


def clip(image_tensor, use_fp16=False):
    '''
    adjust the input based on mean and variance
    '''
    if use_fp16:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float16)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float16)
    else:
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
    for c in range(3):
        m, s = mean[c], std[c]
        image_tensor[:, c] = torch.clamp(image_tensor[:, c], -m / s, (1 - m) / s)
    return image_tensor

## コサイン類似度版

In [ ]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
# targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)
protos_tensor = F.normalize(protos_tensor, dim=1)           # 行方向 L2 正規化

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================


img_original = parameters["resolution"]

torch.manual_seed(777)
random.seed(777)

inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 2000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        # learning rate scheduling
        lr_scheduler(optimizer, iteration_loc, iteration_loc)


        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # forward pass
        optimizer.zero_grad()
        net.zero_grad()

        outputs = net(inputs_jit)

        # # 交差炎とトピー損失の計算
        # logits_all = outputs["logits"]
        # logits = logits_all[:, ::4] 

        # # R_cross classification loss
        # loss = criterion(logits, targets)



        # 特徴量ベースのターゲット損失
        feature = outputs["features"]
        feat = feature.view(feature.size(0), -1)                # 念のため flatten
        feat = F.normalize(feat, dim=1)                         # (bs, D), L2-normalize

        # protos_tensor: (C, D) をループ外で作っておいたものを使う
        sim = torch.matmul(feat, protos_tensor.t())             # (bs, C)

        # targets は「net._protos の key」と同じラベル空間と仮定
        # label2row で「ラベル → protos_tensor の行 index」に変換する
        row_idx = torch.tensor(
            [label2row[int(t.item())] for t in targets],
            device=feat.device,
            dtype=torch.long
        )           

        # 自分のクラスのプロトタイプとの類似度だけ抜き出す
        sim_pos = sim[torch.arange(feat.size(0), device=feat.device), row_idx]

        # 類似度を最大化したいので、マイナスを取って loss にする
        loss = - sim_pos.mean()
        loss_target = loss.item()



        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("main criterion", loss_target)

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))


# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)

num protos: 50 feat dim: 512
------------iteration 10----------
total loss 6.196278095245361
loss_r_feature 402.8768310546875
main criterion -0.43645766377449036
------------iteration 20----------
total loss 4.922033786773682
loss_r_feature 361.7252197265625
main criterion -0.7073180079460144
------------iteration 30----------
total loss 3.502009391784668
loss_r_feature 316.8214111328125
main criterion -0.8355851769447327
------------iteration 40----------
total loss 2.670742988586426
loss_r_feature 283.7657165527344
main criterion -0.8556100726127625
------------iteration 50----------
total loss 2.2490386962890625
loss_r_feature 234.6692657470703
main criterion -0.877985954284668
------------iteration 60----------
total loss 1.9948588609695435
loss_r_feature 195.96873474121094
main criterion -0.8645817041397095
------------iteration 70----------
total loss 1.8643876314163208
loss_r_feature 175.01011657714844
main criterion -0.8602522611618042
------------iteration 80----------
total l

## ユークリッド距離版

In [6]:
# exp_name = args.exp_name
# # final images will be stored here:
# adi_data_path = "./final_images/%s"%exp_name
# # temporal data and generations will be stored here
# exp_name = "generations/%s"%exp_name

iterations = 4000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 120
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"

# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# ver1からr_featureを増加
# coefficients["r_feature"] = 0.03   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# ver3からr_featureを増加
# coefficients["r_feature"] = 0.1   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"


# ver1 から main_loss_multiplie を少し増加
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# ver1 から main_loss_multiplie を少し増加
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v6"

# ver6 から main_loss_multiplie を少し増加
# coefficients["r_feature"] = 0.03   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v7"

# ver7 から l2 を少し増加
# coefficients["r_feature"] = 0.03   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v8"

# ver8 から main_loss_multiplier を少し増加
# coefficients["r_feature"] = 0.03   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 2.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v9"

# ver9 から r_feature を減少
# coefficients["r_feature"] = 0.005   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 2.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v10"

# ver9 から r_feature を増加（結構いい）
# coefficients["r_feature"] = 0.1   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 2.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v11"

# ver11 から r_feature を増加
# coefficients["r_feature"] = 1   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0005
# coefficients["l2"] = 0.00005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 2.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v12"

# ver9 から r_feature を増加（結構いい）
coefficients["r_feature"] = 0.1   # github由来
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.0005
coefficients["l2"] = 0.00005
coefficients["lr"] = 0.25
coefficients["main_loss_multiplier"] = 2.5
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v13"


network_output_function = lambda x: x


prefix = "runs/data_generation_di++_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [9]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
# targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================


img_original = parameters["resolution"]

torch.manual_seed(777)
random.seed(777)

inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 4000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        # learning rate scheduling
        lr_scheduler(optimizer, iteration_loc, iteration_loc)


        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # forward pass
        optimizer.zero_grad()
        net.zero_grad()

        outputs = net(inputs_jit)

        # # 交差炎とトピー損失の計算
        # logits_all = outputs["logits"]
        # logits = logits_all[:, ::4] 

        # # R_cross classification loss
        # loss = criterion(logits, targets)



        # 特徴量ベースのターゲット損失（ユークリッド距離版）
        feature = outputs["features"]
        feat = feature.view(feature.size(0), -1)   # (bs, D)

        # protos_tensor: (C, D) をループ外で作っておいたものを使う
        # targets は net._protos の key と同じラベル空間と仮定
        row_idx = torch.tensor(
            [label2row[int(t.item())] for t in targets],
            device=feat.device,
            dtype=torch.long,
        )

        # 自分のクラスのプロトタイプベクトルを取り出す (bs, D)
        proto_pos = protos_tensor[row_idx]  # (bs, D)

        # ユークリッド距離（二乗）を計算
        # dist_i^2 = ||feat_i - proto_i||^2
        dist2 = torch.sum((feat - proto_pos) ** 2, dim=1)  # (bs,)

        # 平均距離（二乗）を最小化
        loss = dist2.mean()
        loss_target = loss.item()



        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("main criterion", loss_target)

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))


# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)

num protos: 50 feat dim: 512
------------iteration 10----------
total loss 74.60260009765625
loss_r_feature 412.2000427246094
main criterion 12.200281143188477
------------iteration 20----------
total loss 62.950843811035156
loss_r_feature 387.2104187011719
main criterion 8.547005653381348
------------iteration 30----------
total loss 55.27876281738281
loss_r_feature 362.3675231933594
main criterion 6.464333534240723
------------iteration 40----------
total loss 51.871665954589844
loss_r_feature 347.3188171386719
main criterion 5.674283981323242
------------iteration 50----------
total loss 48.758609771728516
loss_r_feature 335.4245910644531
main criterion 4.859089374542236
------------iteration 60----------
total loss 49.35296630859375
loss_r_feature 326.37890625
main criterion 5.399831771850586
------------iteration 70----------
total loss 49.36040496826172
loss_r_feature 324.1311340332031
main criterion 5.436781883239746
------------iteration 80----------
total loss 49.8271560668945

In [54]:
# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)

saved DeepInversion inputs to: runs/data_generation_di++_imnet/debug_imnet100_v12/best_images/deepinversion_inputs.pth


## 交差エントロピー損失＋ユークリッド距離

In [6]:
# exp_name = args.exp_name
# # final images will be stored here:
# adi_data_path = "./final_images/%s"%exp_name
# # temporal data and generations will be stored here
# exp_name = "generations/%s"%exp_name

iterations = 4000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 120
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["main_loss_multiplier_eu"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"


# ver1 から main_loss_multiplier_eu を減少
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["main_loss_multiplier_eu"] = 0.1
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# ver2 から main_loss_multiplier_eu を減少
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["main_loss_multiplier_eu"] = 0.05
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# ver3 から l2 を減少
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.000005
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["main_loss_multiplier_eu"] = 0.05
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"

# ver4 から l2 を減少
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.000001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["main_loss_multiplier_eu"] = 0.05
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# ver5 から tv_l2 を増加
coefficients["r_feature"] = 0.01   # github由来
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.0005
coefficients["l2"] = 0.000001
coefficients["lr"] = 0.25
coefficients["main_loss_multiplier"] = 1.0
coefficients["main_loss_multiplier_eu"] = 0.05
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v6"




network_output_function = lambda x: x


prefix = "runs/data_generation_di+++_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [10]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
# targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================


img_original = parameters["resolution"]

torch.manual_seed(777)
random.seed(777)

inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 4000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        # learning rate scheduling
        lr_scheduler(optimizer, iteration_loc, iteration_loc)


        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # forward pass
        optimizer.zero_grad()
        net.zero_grad()

        outputs = net(inputs_jit)

        # 交差エントロピー損失
        logits_all = outputs["logits"]
        logits = logits_all[:, ::4] 
        loss = criterion(logits, targets)
        loss_target = loss.item()



        # 特徴量ベースのターゲット損失（ユークリッド距離版）
        feature = outputs["features"]
        feat = feature.view(feature.size(0), -1)   # (bs, D)

        # protos_tensor: (C, D) をループ外で作っておいたものを使う
        # targets は net._protos の key と同じラベル空間と仮定
        row_idx = torch.tensor(
            [label2row[int(t.item())] for t in targets],
            device=feat.device,
            dtype=torch.long,
        )

        # 自分のクラスのプロトタイプベクトルを取り出す (bs, D)
        proto_pos = protos_tensor[row_idx]  # (bs, D)

        # ユークリッド距離（二乗）を計算
        # dist_i^2 = ||feat_i - proto_i||^2
        dist2 = torch.sum((feat - proto_pos) ** 2, dim=1)  # (bs,)

        # 平均距離（二乗）を最小化
        loss_eu = dist2.mean()
        loss_target_eu = loss_eu.item()



        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + coefficients["main_loss_multiplier_eu"] * loss_eu + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("main criterion", loss_target)
            print("main eu criterion", loss_eu.item())
            print("loss_var_l2", loss_var_l2.item())
            print("loss_l2", loss_l2)
            print("")

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))


# 最適化した画像を保存
save_dir = os.path.join(prefix, "best_images")  # 画像を保存しているディレクトリと揃える例
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "deepinversion_inputs.pth")

to_save = {
    "images": best_inputs.detach().cpu(),   # 形: (bs, 3, H, W)
    "targets": targets.detach().cpu(),      # 対応するラベル
    "resolution": img_original,
    "classes": targets.unique().tolist(),   # どのクラスを生成したかのメモ（お好みで）
}
torch.save(to_save, save_path)
print("saved DeepInversion inputs to:", save_path)

num protos: 50 feat dim: 512
------------iteration 10----------
total loss 11.179548263549805
loss_r_feature 410.4564514160156
main criterion 3.619096517562866
main eu criterion 13.222429275512695
loss_var_l2 5589.349609375
loss_l2 tensor(91.6014, device='cuda:0', grad_fn=<MeanBackward0>)

------------iteration 20----------
total loss 9.892187118530273
loss_r_feature 380.0848388671875
main criterion 3.0307326316833496
main eu criterion 10.328621864318848
loss_var_l2 5088.17626953125
loss_l2 tensor(86.6722, device='cuda:0', grad_fn=<MeanBackward0>)

------------iteration 30----------
total loss 8.340784072875977
loss_r_feature 352.05743408203125
main criterion 1.902492880821228
main eu criterion 12.286072731018066
loss_var_l2 4606.654296875
loss_l2 tensor(85.4301, device='cuda:0', grad_fn=<MeanBackward0>)

------------iteration 40----------
total loss 7.431879997253418
loss_r_feature 333.1926574707031
main criterion 1.3051049709320068
main eu criterion 14.241674423217773
loss_var_l2 416